# Feature Engineering para MIND Small

Este notebook corresponde al primer avance de la segunda etapa del proyecto **Multimodalidad: recomendación de noticias**. El objetivo es construir una tabla enriquecida de noticias a partir del dataset MIND Small, incorporando variables que permitan avanzar hacia un recomendador sensible a contenido, diversidad, novedad y un proxy inicial de confiabilidad/veracidad.

En la entrega anterior se implementaron modelos baseline como Random, Most Popular y TF-IDF. Sin embargo, el feedback recibido señaló que era necesario especificar con mayor claridad qué features se usarían para representar diversidad y veracidad, de dónde vendrían y cómo serían integradas. Este notebook aborda directamente ese punto mediante la construcción de un archivo de features por noticia.

El procesamiento realizado incluye:

1. **Carga y limpieza de noticias** desde `news.tsv`, manteniendo variables base como `news_id`, `category`, `subcategory`, `title`, `abstract`, URL y entidades.
2. **Features de contenido**, tales como largo del título, largo del resumen, número de palabras y presencia de abstract.
3. **Features basadas en entidades Wikidata**, usando las entidades ya incluidas en MIND para calcular cantidad de entidades en título, abstract y noticia completa, además de la confianza promedio de las entidades detectadas.
4. **Features de popularidad**, calculadas desde `behaviors.tsv`, incluyendo número de impresiones, número de clicks, CTR y popularidad normalizada.
5. **Feature de novedad**, construida a partir de la popularidad histórica de cada noticia. La intuición es que noticias menos expuestas en train son más novedosas.
6. **Proxy inicial de confiabilidad/veracidad**, construido a partir de señales disponibles en el dataset: presencia de abstract, número de entidades, confianza promedio de entidades y bajo score de clickbait.
7. **Label sintético de baja veracidad**, definido como el 20% de noticias con menor `veracity_proxy`. Este label no representa veracidad real, sino una variable sintética para experimentación controlada.

Es importante destacar que MIND no contiene etiquetas reales de noticias falsas o verdaderas. Por lo tanto, la variable `veracity_proxy` no debe interpretarse como una medición real de veracidad, sino como una aproximación inicial de confiabilidad/informatividad que permitirá experimentar con técnicas de re-ranking.

El resultado principal de este notebook son dos archivos:

- `news_features.csv`: archivo completo con features de contenido, entidades, popularidad, novedad y proxy de veracidad.
- `news_features_light.csv`: versión liviana para análisis y modelos posteriores, sin columnas pesadas como listas completas de entidades o abstracts largos.

Estos archivos serán usados en la siguiente etapa para implementar un modelo de re-ranking que ajuste el ranking base de TF-IDF considerando relevancia, diversidad, novedad y confiabilidad.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile
import os
import json
import ast
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances

PROJECT_DIR = Path("/content/drive/MyDrive/Proyecto")

TRAIN_ZIP = PROJECT_DIR / "MINDsmall_train.zip"
DEV_ZIP = PROJECT_DIR / "MINDsmall_dev.zip"

EXTRACT_DIR = PROJECT_DIR / "mindsmall_extracted"
TRAIN_DIR = EXTRACT_DIR / "train"
DEV_DIR = EXTRACT_DIR / "dev"

print("Existe carpeta proyecto:", PROJECT_DIR.exists())
print("Existe zip train:", TRAIN_ZIP.exists())
print("Existe zip dev:", DEV_ZIP.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Existe carpeta proyecto: True
Existe zip train: True
Existe zip dev: True


In [2]:
EXTRACT_DIR.mkdir(exist_ok=True)

def unzip_file(zip_path, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(target_dir)
    print(f"Extraído: {zip_path.name} -> {target_dir}")

unzip_file(TRAIN_ZIP, TRAIN_DIR)
unzip_file(DEV_ZIP, DEV_DIR)

print("TRAIN:")
for f in TRAIN_DIR.rglob("*"):
    if f.is_file():
        print(f)

print("\nDEV:")
for f in DEV_DIR.rglob("*"):
    if f.is_file():
        print(f)

Extraído: MINDsmall_train.zip -> /content/drive/MyDrive/Proyecto/mindsmall_extracted/train
Extraído: MINDsmall_dev.zip -> /content/drive/MyDrive/Proyecto/mindsmall_extracted/dev
TRAIN:
/content/drive/MyDrive/Proyecto/mindsmall_extracted/train/MINDsmall_train/behaviors.tsv
/content/drive/MyDrive/Proyecto/mindsmall_extracted/train/MINDsmall_train/news.tsv
/content/drive/MyDrive/Proyecto/mindsmall_extracted/train/MINDsmall_train/entity_embedding.vec
/content/drive/MyDrive/Proyecto/mindsmall_extracted/train/MINDsmall_train/relation_embedding.vec

DEV:
/content/drive/MyDrive/Proyecto/mindsmall_extracted/dev/MINDsmall_dev/behaviors.tsv
/content/drive/MyDrive/Proyecto/mindsmall_extracted/dev/MINDsmall_dev/news.tsv
/content/drive/MyDrive/Proyecto/mindsmall_extracted/dev/MINDsmall_dev/entity_embedding.vec
/content/drive/MyDrive/Proyecto/mindsmall_extracted/dev/MINDsmall_dev/relation_embedding.vec


A veces al descomprimir queda una subcarpeta adentro. Por eso usaremos una función para buscar los archivos.

In [3]:
def find_file(base_dir, filename):
    matches = list(base_dir.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"No encontré {filename} dentro de {base_dir}")
    return matches[0]

train_news_path = find_file(TRAIN_DIR, "news.tsv")
train_behaviors_path = find_file(TRAIN_DIR, "behaviors.tsv")
train_entity_emb_path = find_file(TRAIN_DIR, "entity_embedding.vec")

dev_news_path = find_file(DEV_DIR, "news.tsv")
dev_behaviors_path = find_file(DEV_DIR, "behaviors.tsv")

print(train_news_path)
print(train_behaviors_path)
print(train_entity_emb_path)

/content/drive/MyDrive/Proyecto/mindsmall_extracted/train/MINDsmall_train/news.tsv
/content/drive/MyDrive/Proyecto/mindsmall_extracted/train/MINDsmall_train/behaviors.tsv
/content/drive/MyDrive/Proyecto/mindsmall_extracted/train/MINDsmall_train/entity_embedding.vec


## Cargar news.tsv

In [4]:
news_cols = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities"
]

# Cargar noticias de train
train_news = pd.read_csv(
    train_news_path,
    sep="\t",
    header=None,
    names=news_cols,
    quoting=3
)

# Cargar noticias de dev
dev_news = pd.read_csv(
    dev_news_path,
    sep="\t",
    header=None,
    names=news_cols,
    quoting=3
)

print("train_news:", train_news.shape)
print("dev_news:", dev_news.shape)

# Unir noticias de train + dev
# Esto permite tener features también para noticias candidatas de dev
news = pd.concat([train_news, dev_news], ignore_index=True)

# Si una noticia aparece tanto en train como en dev, dejamos una sola fila
news = news.drop_duplicates(subset="news_id", keep="first").reset_index(drop=True)

print("news combinado:", news.shape)

display(news.head())
news.info()

# Limpiamos nulos
for col in ["title", "abstract", "category", "subcategory", "url"]:
    news[col] = news[col].fillna("")

news["title_entities"] = news["title_entities"].fillna("[]")
news["abstract_entities"] = news["abstract_entities"].fillna("[]")

train_news: (51282, 8)
dev_news: (42416, 8)
news combinado: (65238, 8)


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65238 entries, 0 to 65237
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   news_id            65238 non-null  object
 1   category           65238 non-null  object
 2   subcategory        65238 non-null  object
 3   title              65238 non-null  object
 4   abstract           61823 non-null  object
 5   url                65238 non-null  object
 6   title_entities     65238 non-null  object
 7   abstract_entities  65238 non-null  object
dtypes: object(8)
memory usage: 4.0+ MB


## Features basicas de contenido

In [5]:
def count_words(text):
    if pd.isna(text) or text == "":
        return 0
    return len(str(text).split())

news_features = news.copy()

news_features["title_length"] = news_features["title"].astype(str).str.len()
news_features["abstract_length"] = news_features["abstract"].astype(str).str.len()

news_features["num_words_title"] = news_features["title"].apply(count_words)
news_features["num_words_abstract"] = news_features["abstract"].apply(count_words)

news_features["has_abstract"] = (news_features["abstract"].astype(str).str.strip() != "").astype(int)

news_features[[
    "news_id", "category", "subcategory", "title",
    "title_length", "abstract_length",
    "num_words_title", "num_words_abstract", "has_abstract"
]].head()

,news_id,category,subcategory,title,title_length,abstract_length,num_words_title,num_words_abstract,has_abstract
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...",70,73,11,12,1
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,29,116,6,19,1
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,63,196,12,36,1
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,59,99,12,21,1
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...",57,179,11,31,1


## Extraer entidades Wikidata

Las columnas de entidades vienen como strings con listas de diccionarios. Por ejemplo, cada entidad puede tener campos como WikidataId, Confidence, etc.

In [6]:
def parse_entities(entity_str):
    """
    Convierte el string de entidades de MIND en una lista de diccionarios.
    Si viene vacío o mal formado, devuelve [].
    """
    if pd.isna(entity_str):
        return []

    entity_str = str(entity_str).strip()
    if entity_str == "" or entity_str == "[]":
        return []

    try:
        return json.loads(entity_str)
    except Exception:
        try:
            return ast.literal_eval(entity_str)
        except Exception:
            return []

def get_wikidata_ids(entities):
    ids = []
    for e in entities:
        if isinstance(e, dict):
            wid = e.get("WikidataId")
            if wid is not None and wid != "":
                ids.append(wid)
    return ids

def get_confidences(entities):
    confs = []
    for e in entities:
        if isinstance(e, dict):
            c = e.get("Confidence")
            if c is not None:
                try:
                    confs.append(float(c))
                except:
                    pass
    return confs

In [7]:
news_features["title_entities_parsed"] = news_features["title_entities"].apply(parse_entities)
news_features["abstract_entities_parsed"] = news_features["abstract_entities"].apply(parse_entities)

news_features["title_entity_ids"] = news_features["title_entities_parsed"].apply(get_wikidata_ids)
news_features["abstract_entity_ids"] = news_features["abstract_entities_parsed"].apply(get_wikidata_ids)

news_features["entity_ids"] = news_features.apply(
    lambda row: list(set(row["title_entity_ids"] + row["abstract_entity_ids"])),
    axis=1
)

news_features["num_title_entities"] = news_features["title_entity_ids"].apply(len)
news_features["num_abstract_entities"] = news_features["abstract_entity_ids"].apply(len)
news_features["num_total_entities"] = news_features["entity_ids"].apply(len)

news_features["title_entity_confidences"] = news_features["title_entities_parsed"].apply(get_confidences)
news_features["abstract_entity_confidences"] = news_features["abstract_entities_parsed"].apply(get_confidences)

def mean_confidence(row):
    confs = row["title_entity_confidences"] + row["abstract_entity_confidences"]
    if len(confs) == 0:
        return 0.0
    return float(np.mean(confs))

news_features["entity_confidence_avg"] = news_features.apply(mean_confidence, axis=1)

news_features[[
    "news_id", "title", "entity_ids",
    "num_title_entities", "num_abstract_entities",
    "num_total_entities", "entity_confidence_avg"
]].head()

,news_id,title,entity_ids,num_title_entities,num_abstract_entities,num_total_entities,entity_confidence_avg
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an...","[Q9682, Q43274, Q80976]",3,0,3,0.99000
1,N19639,50 Worst Habits For Belly Fat,[Q193583],1,1,1,1.00000
2,N61837,The Cost of Trump's Aid Freeze in the Trenches...,[Q212],0,1,1,0.94600
3,N53526,I Was An NBA Wife. Here's How It Affected My M...,[Q155223],0,1,1,1.00000
4,N38324,"How to Get Rid of Skin Tags, According to a De...","[Q371820, Q171171, Q3179593]",1,3,3,0.99975


## Cargar behaviors y calcular probabilidad



In [8]:
behaviors_cols = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions"
]

behaviors = pd.read_csv(
    train_behaviors_path,
    sep="\t",
    header=None,
    names=behaviors_cols
)

behaviors.head()

#Ahora calculamos para cada noticia cuántas veces apareció y cuántos clicks tuvo.

def parse_impressions(impression_str):
    """
    Convierte una línea tipo:
    N123-0 N456-1 N789-0
    en pares:
    [(N123, 0), (N456, 1), (N789, 0)]
    """
    if pd.isna(impression_str):
        return []

    pairs = []
    for item in str(impression_str).split():
        if "-" not in item:
            continue
        news_id, clicked = item.rsplit("-", 1)
        try:
            clicked = int(clicked)
        except:
            clicked = 0
        pairs.append((news_id, clicked))
    return pairs


In [9]:
from collections import defaultdict

impressions_count = defaultdict(int)
clicks_count = defaultdict(int)

for imp_str in behaviors["impressions"]:
    for news_id, clicked in parse_impressions(imp_str):
        impressions_count[news_id] += 1
        clicks_count[news_id] += clicked

popularity_df = pd.DataFrame({
    "news_id": list(impressions_count.keys()),
    "impressions_count": [impressions_count[nid] for nid in impressions_count.keys()],
    "clicks_count": [clicks_count[nid] for nid in impressions_count.keys()]
})

popularity_df["ctr"] = popularity_df["clicks_count"] / popularity_df["impressions_count"]

popularity_df.head()

,news_id,impressions_count,clicks_count,ctr
0,N55689,18315,4316,0.235654
1,N35729,15418,3346,0.217019
2,N20678,7059,578,0.081881
3,N39317,6344,517,0.081494
4,N58114,8332,175,0.021003


Último con las noticias

In [10]:
news_features = news_features.merge(
    popularity_df,
    on="news_id",
    how="left"
)

news_features["impressions_count"] = news_features["impressions_count"].fillna(0).astype(int)
news_features["clicks_count"] = news_features["clicks_count"].fillna(0).astype(int)
news_features["ctr"] = news_features["ctr"].fillna(0.0)

news_features[[
    "news_id", "title", "impressions_count", "clicks_count", "ctr"
]].sort_values("impressions_count", ascending=False).head(10)

,news_id,title,impressions_count,clicks_count,ctr
31221,N47061,105 Black Friday Deals You Can Start Shopping ...,23037,820,0.035595
41883,N51048,Rep. Tim Ryan endorses Biden in Democratic pri...,19242,1875,0.097443
31748,N26262,Celebrity plastic surgery transformations,19106,1139,0.059615
5118,N50872,50 amazing gifts for every type of person and ...,18702,279,0.014918
33899,N55689,"Charles Rogers, former Michigan State football...",18315,4316,0.235654
46821,N38779,'One in a million' deer captured on camera in ...,18101,1490,0.082316
50470,N62360,The son of a Chinese billionaire has been bann...,16869,1594,0.094493
43038,N13907,4 reasons you shouldn't max out your 401(k),16267,613,0.037684
48033,N49180,Why Prince Harry Wore His Remembrance Poppy Di...,16022,1103,0.068843
48697,N41020,Lamar Odom Is Engaged to Sabrina Parr: See Her...,15551,1139,0.073243


## Crear popularity_score y novelty_score

In [11]:
# Popularidad normalizada entre 0 y 1
max_impressions = news_features["impressions_count"].max()

if max_impressions > 0:
    news_features["popularity_score"] = news_features["impressions_count"] / max_impressions
else:
    news_features["popularity_score"] = 0.0

# Novedad: noticias menos vistas tienen mayor novelty
total_impressions = news_features["impressions_count"].sum()

news_features["novelty_score"] = -np.log(
    (news_features["impressions_count"] + 1) / (total_impressions + 1)
)

# Normalizamos novelty entre 0 y 1 para que sea más fácil usarla después
min_nov = news_features["novelty_score"].min()
max_nov = news_features["novelty_score"].max()

if max_nov > min_nov:
    news_features["novelty_score_norm"] = (
        (news_features["novelty_score"] - min_nov) / (max_nov - min_nov)
    )
else:
    news_features["novelty_score_norm"] = 0.0

news_features[[
    "news_id", "impressions_count", "popularity_score", "novelty_score_norm"
]].head()

,news_id,impressions_count,popularity_score,novelty_score_norm
0,N55528,0,0.000000,1.000000
1,N19639,0,0.000000,1.000000
2,N61837,0,0.000000,1.000000
3,N53526,0,0.000000,1.000000
4,N38324,1,0.000043,0.930995


## Crear un clickbait_score simple

Esto será una proxy inicial, no una detección real.

In [12]:
## ANTIGUO


#CLICKBAIT_PATTERNS = [
#    r"you won't believe",
#    r"won't believe",
#    r"shocking",
#    r"what happens next",
#    r"this is why",
#    r"will blow your mind",
#    r"secret",
#    r"revealed",
#    r"never seen",
#    r"can't believe",
#    r"\d+\s+(things|ways|reasons|tips)"
#]
#
#def clickbait_score(title):
#    title = str(title).lower()
#
#    score = 0
#
#    # Patrones lingüísticos
#    for pattern in CLICKBAIT_PATTERNS:
#        if re.search(pattern, title):
#            score += 1
#
#    # Signos de exclamación
#    if "!" in title:
#        score += 1
#
#    # Título muy interrogativo
#    if title.count("?") >= 2:
#        score += 1
#
#    # Muchas mayúsculas respecto del largo
#    original = str(title)
#    if len(original) > 0:
#        uppercase_ratio = sum(1 for c in original if c.isupper()) / len(original)
#        if uppercase_ratio > 0.4:
#            score += 1
#
#    # Lo dejamos entre 0 y 1
#    return min(score / 3, 1.0)
#
#news_features["clickbait_score"] = news_features["title"].apply(clickbait_score)
#
#news_features[["news_id", "title", "clickbait_score"]].sort_values(
#    "clickbait_score", ascending=False
#).head(20)

In [13]:
CLICKBAIT_PATTERNS = [
    r"you won't believe",
    r"won't believe",
    r"shocking",
    r"what happens next",
    r"happens next",
    r"this is why",
    r"here'?s why",
    r"here is why",
    r"the reason why",
    r"what you need to know",
    r"things you didn'?t know",
    r"will blow your mind",
    r"will shock you",
    r"secret",
    r"revealed",
    r"never seen",
    r"can't believe",
    r"must see",
    r"goes viral",
    r"find out",
    r"you need to see",
    r"this is what",
    r"everything you need to know",
    r"what we know",
    r"what to know",
    r"what happened",
    r"why you should",
    r"don'?t miss",
    r"exclusive",
    r"unbelievable",
    r"jaw[- ]dropping",
    r"mind[- ]blowing",
    r"\d+\s+(things|ways|reasons|tips|facts|secrets|tricks)"
]




titles = news_features["title"]
abstracts = news_features["abstract"]

#Añadimos una comparacion de titulo con abstract
texts_for_tfidf = pd.concat([titles, abstracts], ignore_index=True)

vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(texts_for_tfidf)

n = len(news_features)
title_vecs = tfidf_matrix[:n]
abstract_vecs = tfidf_matrix[n:]

#usamos similaridad coseno
cosine_sim = 1 - paired_cosine_distances(title_vecs, abstract_vecs)
cosine_sim = np.nan_to_num(cosine_sim, nan=0.0, posinf=0.0, neginf=0.0)
cosine_sim = np.clip(cosine_sim, 0, 1)


#calculamos un índice de diferencia entre abstract y título,
#entre mas alto mas clickbait
news_features["title_abstract_mismatch"] = np.where(
    news_features["has_abstract"],
    1 - cosine_sim,
    0.0
)

def clickbait_score(title, title_abstract_mismatch=0.0):
    original = "" if pd.isna(title) else str(title)
    title = original.lower()

    score = 0

    #Patrones
    for pattern in CLICKBAIT_PATTERNS:
        if re.search(pattern, title):
            score += 1

    # Signos de exclamación
    if "!" in title:
        score += 1

    # Título muy interrogativo
    if title.count("?") >= 2:
        score += 1

    # Muchas mayúsculas respecto del largo
    if len(original) > 0:
        uppercase_ratio = sum(1 for c in original if c.isupper()) / len(original)
        if uppercase_ratio > 0.4:
            score += 1

    # Comparación título-abstract:
    # si el título se parece poco al abstract, se suma una penalización continua.
    # Si no hay abstract, title_abstract_mismatch vale 0.
    score += float(title_abstract_mismatch)

    # Lo dejamos entre 0 y 1
    return min(score / 4, 1.0)

news_features["clickbait_score"] = news_features.apply(
    lambda row: clickbait_score(
        row["title"],
        row["title_abstract_mismatch"]
    ),
    axis=1
)

news_features[
    ["news_id", "title", "abstract", "title_abstract_mismatch", "clickbait_score"]
].sort_values(
    "clickbait_score", ascending=False
).head(20)

,news_id,title,abstract,title_abstract_mismatch,clickbait_score
52644,N11524,Exclusive: What Can You Fit in the 2020 Corvet...,Testing both the trunk and the frunk,1.000000,1.000000
28331,N51344,'This Is Us' Stars Mandy Moore and Milo Ventim...,ET spoke with the stars about how they spend t...,0.882295,0.970574
12546,N49065,"What happened with Brexit on Super Saturday, a...",Boris Johnson's plans to hold a decisve vote o...,0.877772,0.969443
9137,N48119,What Happened At UFC 244 Last Night?!?,"UFC 244 results: ""Diaz vs. Masvidal"" brought t...",0.737278,0.934319
53688,N52456,Wake up! Here's why you shouldn't grab that ex...,"Sunday morning, daylight saving time bestows i...",0.731615,0.932904
47768,N56284,What Happened At UFC 'Moscow' Last Night?!?,"UFC Fight Night 163 results: ""Zabit vs. Kattar...",0.704658,0.926165
21658,N51961,Is That Picture Real? How Old Is It? The Easy ...,It's easy to see how folks would just assume t...,0.660298,0.915075
29938,N3521,What Happened At UFC 'Singapore' Yesterday?!?,"UFC on Fight Night 162 results: ""Maia vs. Askr...",0.583600,0.895900
40422,N45449,12 secrets behind Walt Disney World's magical ...,From how many lights it takes to make Cinderel...,1.000000,0.750000
50444,N27125,Happy Friday! What if You Always Had It Off? W...,The reasons that a four-day workweek hasn't ye...,1.000000,0.750000


## Crear veracity_proxy

Normalizamos cantidad de entidades y confianza.

In [14]:
def minmax(series):
    min_val = series.min()
    max_val = series.max()
    if max_val > min_val:
        return (series - min_val) / (max_val - min_val)
    else:
        return pd.Series(0.0, index=series.index)

news_features["num_total_entities_norm"] = minmax(news_features["num_total_entities"])
news_features["entity_confidence_norm"] = minmax(news_features["entity_confidence_avg"])

Ahora el proxy

In [15]:
news_features["veracity_proxy"] = (
    0.35 * news_features["has_abstract"] +
    0.30 * news_features["num_total_entities_norm"] +
    0.20 * news_features["entity_confidence_norm"] +
    0.15 * (1 - news_features["clickbait_score"])
)

# Aseguramos rango [0, 1]
news_features["veracity_proxy"] = news_features["veracity_proxy"].clip(0, 1)

news_features[[
    "news_id", "title", "has_abstract", "num_total_entities",
    "entity_confidence_avg", "clickbait_score", "veracity_proxy"
]].sort_values("veracity_proxy", ascending=False).head(10)


# IMPORTANTE: esto no mide verdad real. Es un proxy de confiabilidad/informatividad.

,news_id,title,has_abstract,num_total_entities,entity_confidence_avg,clickbait_score,veracity_proxy
26809,N1227,Oscars: These Are the 42 Films Vying for Best ...,1,30,0.997281,0.202257,0.969118
53221,N21743,Milwaukee tops list of hottest travel destinat...,1,17,0.996167,0.099834,0.854258
18242,N47479,'Yabba Dabba Doo!' Kimye and Kids Rock Flintst...,1,22,0.987208,0.450947,0.849800
33124,N54310,Weather warnings and advisories posted as cold...,1,18,0.972611,0.192981,0.845575
11617,N30682,"Rock Hall of Fame: Notorious B.I.G., Whitney H...",1,17,0.999810,0.172263,0.844122
11687,N39740,President Donald Trump coming to South Carolina,1,17,0.992611,0.221196,0.835343
7295,N3137,NWS issues Flash Flood Watch for several Mid-S...,1,16,0.994059,0.189225,0.830428
54993,N62143,Cleveland Browns vs. New England Patriots: How...,1,16,0.998222,0.201569,0.829409
52535,N57793,Tornado watch issued for much of Maryland unti...,1,15,0.992125,0.151008,0.825774
42197,N61877,Ford EcoBoost 400 Entry List: Homestead-Miami ...,1,16,0.999938,0.229251,0.825600


##Crear un label sintético opcional de baja veracidad

Esto sirve para evaluar después si el re-ranking reduce noticias “sospechosas” según la proxy.

In [16]:
threshold = news_features["veracity_proxy"].quantile(0.20)

news_features["low_veracity_synthetic"] = (
    news_features["veracity_proxy"] <= threshold
).astype(int)

news_features["low_veracity_synthetic"].value_counts(normalize=True)

,proportion
low_veracity_synthetic,
0,0.799994
1,0.200006


## Guardar news_features.csv

Antes de guardar, eliminamos columnas pesadas o poco prácticas.

In [17]:
columns_to_save = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "abstract",
    "url",

    "title_length",
    "abstract_length",
    "num_words_title",
    "num_words_abstract",
    "has_abstract",

    "title_entity_ids",
    "abstract_entity_ids",
    "entity_ids",
    "num_title_entities",
    "num_abstract_entities",
    "num_total_entities",
    "entity_confidence_avg",

    "impressions_count",
    "clicks_count",
    "ctr",
    "popularity_score",
    "novelty_score",
    "novelty_score_norm",

    "clickbait_score",
    "veracity_proxy",
    "low_veracity_synthetic"
]

final_features = news_features[columns_to_save].copy()

output_path = PROJECT_DIR / "news_features.csv"
final_features.to_csv(output_path, index=False)

print("Guardado en:", output_path)
print(final_features.shape)
final_features.head()

Guardado en: /content/drive/MyDrive/Proyecto/news_features.csv
(65238, 27)


,news_id,category,subcategory,title,abstract,url,title_length,abstract_length,num_words_title,num_words_abstract,...,entity_confidence_avg,impressions_count,clicks_count,ctr,popularity_score,novelty_score,novelty_score_norm,clickbait_score,veracity_proxy,low_veracity_synthetic
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,70,73,11,12,...,0.99000,0,0,0.0,0.000000,15.580831,1.000000,0.250000,0.690500,0
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,29,116,6,19,...,1.00000,0,0,0.0,0.000000,15.580831,1.000000,0.130886,0.690367,0
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,63,196,12,36,...,0.94600,0,0,0.0,0.000000,15.580831,1.000000,0.223696,0.665646,1
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,59,99,12,21,...,1.00000,0,0,0.0,0.000000,15.580831,1.000000,0.185266,0.682210,0
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,57,179,11,31,...,0.99975,1,0,0.0,0.000043,14.887684,0.930995,0.087169,0.716875,0


## Revisiones rápidas para saber si quedó bien

In [18]:
#Revisar tamaño y primeras filas
print("Shape final_features:", final_features.shape)
final_features.head()

Shape final_features: (65238, 27)


,news_id,category,subcategory,title,abstract,url,title_length,abstract_length,num_words_title,num_words_abstract,...,entity_confidence_avg,impressions_count,clicks_count,ctr,popularity_score,novelty_score,novelty_score_norm,clickbait_score,veracity_proxy,low_veracity_synthetic
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,70,73,11,12,...,0.99000,0,0,0.0,0.000000,15.580831,1.000000,0.250000,0.690500,0
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,29,116,6,19,...,1.00000,0,0,0.0,0.000000,15.580831,1.000000,0.130886,0.690367,0
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,63,196,12,36,...,0.94600,0,0,0.0,0.000000,15.580831,1.000000,0.223696,0.665646,1
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,59,99,12,21,...,1.00000,0,0,0.0,0.000000,15.580831,1.000000,0.185266,0.682210,0
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,57,179,11,31,...,0.99975,1,0,0.0,0.000043,14.887684,0.930995,0.087169,0.716875,0


In [19]:
#Revisar columnas disponibles
final_features.columns.tolist()

['news_id',
 'category',
 'subcategory',
 'title',
 'abstract',
 'url',
 'title_length',
 'abstract_length',
 'num_words_title',
 'num_words_abstract',
 'has_abstract',
 'title_entity_ids',
 'abstract_entity_ids',
 'entity_ids',
 'num_title_entities',
 'num_abstract_entities',
 'num_total_entities',
 'entity_confidence_avg',
 'impressions_count',
 'clicks_count',
 'ctr',
 'popularity_score',
 'novelty_score',
 'novelty_score_norm',
 'clickbait_score',
 'veracity_proxy',
 'low_veracity_synthetic']

In [20]:
#Estadisticas solo de columnas numericas
numeric_cols = final_features.select_dtypes(include=["number"]).columns

numeric_summary = final_features[numeric_cols].describe().T
numeric_summary



,count,mean,std,min,25%,50%,75%,max
title_length,65238.0,66.275453,19.105746,11.000000,54.000000,64.000000,78.000000,251.000000
abstract_length,65238.0,208.198780,158.370804,0.000000,90.000000,146.000000,393.000000,2603.000000
num_words_title,65238.0,10.736488,3.241998,1.000000,9.000000,10.000000,13.000000,48.000000
num_words_abstract,65238.0,34.859131,26.626443,0.000000,15.000000,25.000000,64.000000,474.000000
has_abstract,65238.0,0.947653,0.222727,0.000000,1.000000,1.000000,1.000000,1.000000
num_title_entities,65238.0,1.180830,0.974322,0.000000,0.000000,1.000000,2.000000,8.000000
num_abstract_entities,65238.0,1.894862,1.838848,0.000000,0.000000,1.000000,3.000000,30.000000
num_total_entities,65238.0,2.384822,1.839299,0.000000,1.000000,2.000000,3.000000,30.000000
entity_confidence_avg,65238.0,0.868238,0.329065,0.000000,0.983333,0.997500,1.000000,1.000000
impressions_count,65238.0,89.571170,662.271563,0.000000,0.000000,0.000000,2.000000,23037.000000


In [21]:
#Revisar columnas clave
key_cols = [
    "news_id",
    "category",
    "subcategory",
    "num_total_entities",
    "impressions_count",
    "clicks_count",
    "ctr",
    "popularity_score",
    "novelty_score_norm",
    "clickbait_score",
    "veracity_proxy",
    "low_veracity_synthetic"
]

final_features[key_cols].head(10)

,news_id,category,subcategory,num_total_entities,impressions_count,clicks_count,ctr,popularity_score,novelty_score_norm,clickbait_score,veracity_proxy,low_veracity_synthetic
0,N55528,lifestyle,lifestyleroyals,3,0,0,0.0,0.000000,1.000000,0.250000,0.690500,0
1,N19639,health,weightloss,1,0,0,0.0,0.000000,1.000000,0.130886,0.690367,0
2,N61837,news,newsworld,1,0,0,0.0,0.000000,1.000000,0.223696,0.665646,1
3,N53526,health,voices,1,0,0,0.0,0.000000,1.000000,0.185266,0.682210,0
4,N38324,health,medical,3,1,0,0.0,0.000043,0.930995,0.087169,0.716875,0
5,N2073,sports,football_nfl,1,0,0,0.0,0.000000,1.000000,0.106041,0.694094,0
6,N49186,weather,weathertopstories,1,0,0,0.0,0.000000,1.000000,0.226212,0.668468,1
7,N59295,news,newsworld,2,0,0,0.0,0.000000,1.000000,0.136357,0.697613,0
8,N24510,entertainment,gaming,1,0,0,0.0,0.000000,1.000000,0.169593,0.684561,0
9,N39237,news,newsscienceandtechnology,2,3,0,0.0,0.000130,0.861990,0.209833,0.688225,0


In [22]:
#Resumen
print("Noticias:", len(final_features))
print("Categorías únicas:", final_features["category"].nunique())
print("Subcategorías únicas:", final_features["subcategory"].nunique())

print("\nPromedio entidades por noticia:")
print(final_features["num_total_entities"].mean())

print("\nPromedio veracity_proxy:")
print(final_features["veracity_proxy"].mean())

print("\nPorcentaje low_veracity_synthetic:")
print(final_features["low_veracity_synthetic"].mean())

print("\nPromedio novelty_score_norm:")
print(final_features["novelty_score_norm"].mean())

Noticias: 65238
Categorías únicas: 18
Subcategorías únicas: 270

Promedio entidades por noticia:
2.3848217296667586

Promedio veracity_proxy:
0.6532533645645119

Porcentaje low_veracity_synthetic:
0.20000613139581225

Promedio novelty_score_norm:
0.9153940504852778


In [23]:
#Distribucion por categoria
category_summary = (
    final_features
    .groupby("category")
    .agg(
        num_news=("news_id", "count"),
        avg_veracity_proxy=("veracity_proxy", "mean"),
        avg_novelty=("novelty_score_norm", "mean"),
        avg_clickbait=("clickbait_score", "mean"),
        avg_entities=("num_total_entities", "mean"),
        avg_ctr=("ctr", "mean")
    )
    .sort_values("num_news", ascending=False)
)

category_summary

,num_news,avg_veracity_proxy,avg_novelty,avg_clickbait,avg_entities,avg_ctr
category,,,,,,
news,20039,0.666852,0.918707,0.167661,2.505015,0.012076
sports,19368,0.661501,0.935603,0.165856,2.852902,0.017464
finance,3786,0.638940,0.896214,0.187619,1.727681,0.010572
foodanddrink,3123,0.634737,0.899021,0.193664,1.613513,0.010452
travel,3013,0.644562,0.904117,0.172139,2.164620,0.012674
lifestyle,2991,0.602927,0.876772,0.192610,1.429957,0.010965
video,2712,0.671462,0.920526,0.181728,2.539454,0.013132
weather,2601,0.645898,0.912804,0.166426,2.465590,0.011786
health,2207,0.589957,0.883997,0.202510,1.085637,0.010247


In [24]:
#Top noticias populares
final_features[[
    "news_id",
    "title",
    "category",
    "impressions_count",
    "clicks_count",
    "ctr",
    "veracity_proxy",
    "novelty_score_norm"
]].sort_values("impressions_count", ascending=False).head(20)

,news_id,title,category,impressions_count,clicks_count,ctr,veracity_proxy,novelty_score_norm
31221,N47061,105 Black Friday Deals You Can Start Shopping ...,lifestyle,23037,820,0.035595,0.776095,0.000000
41883,N51048,Rep. Tim Ryan endorses Biden in Democratic pri...,news,19242,1875,0.097443,0.381650,0.017919
31748,N26262,Celebrity plastic surgery transformations,entertainment,19106,1139,0.059615,0.722500,0.018625
5118,N50872,50 amazing gifts for every type of person and ...,lifestyle,18702,279,0.014918,0.468606,0.020753
33899,N55689,"Charles Rogers, former Michigan State football...",sports,18315,4316,0.235654,0.714388,0.022834
46821,N38779,'One in a million' deer captured on camera in ...,news,18101,1490,0.082316,0.692435,0.024004
50470,N62360,The son of a Chinese billionaire has been bann...,news,16869,1594,0.094493,0.684484,0.031022
43038,N13907,4 reasons you shouldn't max out your 401(k),finance,16267,613,0.037684,0.437263,0.034639
48033,N49180,Why Prince Harry Wore His Remembrance Poppy Di...,lifestyle,16022,1103,0.068843,0.689686,0.036150
48697,N41020,Lamar Odom Is Engaged to Sabrina Parr: See Her...,entertainment,15551,1139,0.073243,0.672500,0.039120


In [25]:
#Top noticias mayor CTR
min_impressions = 20

final_features.loc[
    final_features["impressions_count"] >= min_impressions,
    [
        "news_id",
        "title",
        "category",
        "impressions_count",
        "clicks_count",
        "ctr",
        "veracity_proxy",
        "novelty_score_norm"
    ]
].sort_values("ctr", ascending=False).head(20)

,news_id,title,category,impressions_count,clicks_count,ctr,veracity_proxy,novelty_score_norm
45900,N3462,Missing Lakeview woman's body recovered in Aug...,news,20,8,0.400000,0.689391,0.696909
42756,N22696,Free Fight! Blachowicz Flatlines Rockhold In V...,sports,46,18,0.391304,0.668422,0.616706
37599,N25543,Hillary Clinton: 'I'm under enormous pressure'...,news,21,8,0.380952,0.683695,0.692277
50365,N49279,Broadway Actress Laurel Griggs Dies at Age 13,music,6229,2270,0.364424,0.702932,0.130192
37026,N60750,"Browns, Steelers brawl at end of Cleveland's 2...",sports,353,125,0.354108,0.721146,0.415694
37370,N35139,Has Hargreaves sealed his fate with the Buccan...,sports,22,7,0.318182,0.682400,0.687852
50800,N50387,Diaz believes Masvidal was on the verge of giv...,sports,22,7,0.318182,0.695091,0.687852
50714,N49685,Broadway Star Laurel Griggs Suffered Asthma At...,music,7229,2294,0.317333,0.703336,0.115373
45974,N53585,"Rip Taylor's Cause of Death Revealed, Memorial...",tv,9908,2835,0.286132,0.638314,0.083993
35063,N39463,"Jordan Spieth, Henrik Stenson round out Tiger ...",sports,42,12,0.285714,0.733471,0.625561


In [26]:
#Menor veracity proxy
final_features[[
    "news_id",
    "title",
    "category",
    "has_abstract",
    "num_total_entities",
    "entity_confidence_avg",
    "clickbait_score",
    "veracity_proxy",
    "low_veracity_synthetic"
]].sort_values("veracity_proxy", ascending=True).head(20)

,news_id,title,category,has_abstract,num_total_entities,entity_confidence_avg,clickbait_score,veracity_proxy,low_veracity_synthetic
34028,N10372,"New Gas Tax? Here's Why VA, DC Could Pay More ...",news,0,0,0.0,0.25,0.1125,1
44291,N41416,5 reasons to watch Tailgate 19,sports,0,0,0.0,0.25,0.1125,1
52021,N19371,"A surprise party, Sriracha secrets and",travel,0,0,0.0,0.25,0.1125,1
48853,N49684,9 facts about EDC Orlando,travel,0,0,0.0,0.25,0.1125,1
53617,N34717,Breezy and cooler - Hello fall!,weather,0,0,0.0,0.25,0.1125,1
20024,N6781,Trolling From an Aircraft Carrier!,sports,0,0,0.0,0.25,0.1125,1
36821,N50986,StormTALK! Weather Blog: Thursday Edition,weather,0,0,0.0,0.25,0.1125,1
959,N47487,5 reasons to watch Tailgate 19,sports,0,0,0.0,0.25,0.1125,1
48020,N56826,Kente Korner Podcast: Episode 17!,sports,0,0,0.0,0.25,0.1125,1
53224,N45666,FINNEY'S FRIDAY FREE STUFF: Nirvana Bars recov...,foodanddrink,0,0,0.0,0.25,0.1125,1


In [27]:
#Mayor veracity proxy
final_features[[
    "news_id",
    "title",
    "category",
    "has_abstract",
    "num_total_entities",
    "entity_confidence_avg",
    "clickbait_score",
    "veracity_proxy",
    "low_veracity_synthetic"
]].sort_values("veracity_proxy", ascending=False).head(20)

,news_id,title,category,has_abstract,num_total_entities,entity_confidence_avg,clickbait_score,veracity_proxy,low_veracity_synthetic
26809,N1227,Oscars: These Are the 42 Films Vying for Best ...,movies,1,30,0.997281,0.202257,0.969118,0
53221,N21743,Milwaukee tops list of hottest travel destinat...,travel,1,17,0.996167,0.099834,0.854258,0
18242,N47479,'Yabba Dabba Doo!' Kimye and Kids Rock Flintst...,tv,1,22,0.987208,0.450947,0.849800,0
33124,N54310,Weather warnings and advisories posted as cold...,news,1,18,0.972611,0.192981,0.845575,0
11617,N30682,"Rock Hall of Fame: Notorious B.I.G., Whitney H...",music,1,17,0.999810,0.172263,0.844122,0
11687,N39740,President Donald Trump coming to South Carolina,news,1,17,0.992611,0.221196,0.835343,0
7295,N3137,NWS issues Flash Flood Watch for several Mid-S...,weather,1,16,0.994059,0.189225,0.830428,0
54993,N62143,Cleveland Browns vs. New England Patriots: How...,sports,1,16,0.998222,0.201569,0.829409,0
52535,N57793,Tornado watch issued for much of Maryland unti...,news,1,15,0.992125,0.151008,0.825774,0
42197,N61877,Ford EcoBoost 400 Entry List: Homestead-Miami ...,sports,1,16,0.999938,0.229251,0.825600,0


In [28]:
#Menor clickbait_score
final_features[[
    "news_id",
    "title",
    "category",
    "has_abstract",
    "num_total_entities",
    "entity_confidence_avg",
    "clickbait_score",
    "veracity_proxy",
    "low_veracity_synthetic"
]].sort_values("clickbait_score", ascending=True).head(20)

,news_id,title,category,has_abstract,num_total_entities,entity_confidence_avg,clickbait_score,veracity_proxy,low_veracity_synthetic
60746,N7259,Bankrupt TicketRoar leaves thousands of school...,news,0,0,0.000000,0.0,0.150000,1
42802,N57721,Cincinnati Zoo reveals new bearcat's name,news,0,1,0.999000,0.0,0.359800,1
16060,N58658,Water bottle thrown at Conor McGregor in Russi...,sports,0,3,0.999667,0.0,0.379933,1
16056,N19303,Man arrested for pointing gun at church congre...,news,0,0,0.000000,0.0,0.150000,1
60703,N33442,IU quarterback Penix is out for the rest of th...,sports,0,0,0.000000,0.0,0.150000,1
60716,N3218,"Election results in OH, KY, IN; See them here",news,0,0,0.000000,0.0,0.150000,1
22681,N3389,We can now talk about how good DK Metcalf has ...,sports,0,0,0.000000,0.0,0.150000,1
26898,N29219,Schotty's offense is getting it done on early ...,sports,0,0,0.000000,0.0,0.150000,1
60724,N51913,Solar group solicits campaign cash for top law...,news,0,0,0.000000,0.0,0.150000,1
28986,N44083,Jeff Duncan: Saints are best they've ever been...,sports,0,1,0.989000,0.0,0.357800,1


In [29]:
#Mayor clickbait_score
final_features[[
    "news_id",
    "title",
    "category",
    "has_abstract",
    "num_total_entities",
    "entity_confidence_avg",
    "clickbait_score",
    "veracity_proxy",
    "low_veracity_synthetic"
]].sort_values("clickbait_score", ascending=False).head(20)

,news_id,title,category,has_abstract,num_total_entities,entity_confidence_avg,clickbait_score,veracity_proxy,low_veracity_synthetic
52644,N11524,Exclusive: What Can You Fit in the 2020 Corvet...,autos,1,1,1.000000,1.000000,0.560000,1
28331,N51344,'This Is Us' Stars Mandy Moore and Milo Ventim...,tv,1,1,0.999000,0.970574,0.564214,1
12546,N49065,"What happened with Brexit on Super Saturday, a...",news,1,3,0.999750,0.969443,0.584534,1
9137,N48119,What Happened At UFC 244 Last Night?!?,sports,1,5,0.984600,0.934319,0.606772,1
53688,N52456,Wake up! Here's why you shouldn't grab that ex...,news,1,1,0.985000,0.932904,0.567064,1
47768,N56284,What Happened At UFC 'Moscow' Last Night?!?,sports,1,6,0.999000,0.926165,0.620875,1
21658,N51961,Is That Picture Real? How Old Is It? The Easy ...,lifestyle,1,3,0.994333,0.915075,0.591605,1
29938,N3521,What Happened At UFC 'Singapore' Yesterday?!?,sports,1,8,0.989875,0.895900,0.643590,1
40422,N45449,12 secrets behind Walt Disney World's magical ...,travel,1,2,1.000000,0.750000,0.607500,1
50444,N27125,Happy Friday! What if You Always Had It Off? W...,finance,1,0,0.000000,0.750000,0.387500,1


In [30]:
#Guardar version liviana para trabajar despues
light_columns = [
    "news_id",
    "category",
    "subcategory",
    "title",
    "title_length",
    "abstract_length",
    "num_words_title",
    "num_words_abstract",
    "has_abstract",
    "num_title_entities",
    "num_abstract_entities",
    "num_total_entities",
    "entity_confidence_avg",
    "impressions_count",
    "clicks_count",
    "ctr",
    "popularity_score",
    "novelty_score_norm",
    "clickbait_score",
    "veracity_proxy",
    "low_veracity_synthetic"
]

light_features = final_features[light_columns].copy()

light_output_path = PROJECT_DIR / "news_features_light_all.csv"
light_features.to_csv(light_output_path, index=False)

print("Guardado en:", light_output_path)
print("Shape light_features:", light_features.shape)

Guardado en: /content/drive/MyDrive/Proyecto/news_features_light_all.csv
Shape light_features: (65238, 21)
